# Qdrant Essentials: Day 3 - Building Hybrid Search in Qdrant

Let's see how hybrid search might be implemented with Qdrant's Universal Query API.

## Step 1: Install the dependencies

In [1]:
pip install -q qdrant-client[fastembed]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from qdrant_client import QdrantClient, models

c:\Users\Praneeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Connect to Qdrant

Let's connect to a running [Qdrant Cloud](https://cloud.qdrant.io/) cluster and create a collection containing both sparse and dense named vectors.

In [4]:
# from google.colab import userdata
import os
from dotenv import load_dotenv


load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

In [5]:
from qdrant_client import models

# Define the collection name
collection_name = "hybrid_search_demo"

# Create our collection with both sparse (bm25) and dense vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
            distance=models.Distance.COSINE,
            size=384,
        ),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

True

## Step 3: Upload the data into the collection

Now, we have a collection that allows us to store two vectors per point, and we can finally fill it with data.

In [6]:
documents = [
    "Aged Gouda develops a crystalline texture and nutty flavor profile after 18 months of maturation.",
    "Mature Gouda cheese becomes grainy and develops a rich, buttery taste with extended aging.",
    "Brie cheese features a soft, creamy interior surrounded by an edible white rind.",
    "This French cheese has a flowing, buttery center encased in a bloomy white crust.",
    "Fresh mozzarella pairs beautifully with ripe tomatoes and basil leaves.",
    "Classic Margherita pizza topped with tomato sauce, mozzarella, and fresh basil.",
    "Parmesan requires at least 12 months of cave aging to develop its signature sharp taste.",
    "Parmigiano-Reggiano's distinctive piquant flavor comes from extended maturation in controlled environments.",
    "Grilled cheese sandwiches are the ultimate American comfort food for cold winter days.",
    "Croque Monsieur combines ham and Gruyère in France's answer to the toasted cheese sandwich.",
]

In [7]:
import uuid

client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "dense": models.Document(
                    text=doc,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                "sparse": models.Document(
                    text=doc,
                    model="Qdrant/bm25",
                ),
            },
            payload={"text": doc},
        )
        for i, doc in enumerate(documents)
    ]
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\Praneeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Praneeth\AppData\Local\Temp\fastembed_cache\models--qdrant--all-MiniLM-L6-v2-onnx. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 5 f

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

## Step 4: Validating the outputs of sparse and dense search

Both of our models may return completely different sets of results for the same query. Let's check if that's the case.

In [8]:
def dense_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="sentence-transformers/all-MiniLM-L6-v2",
        ),
        using="dense",
        limit=3,
    )
    return response.points

In [9]:
def sparse_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="Qdrant/bm25",
        ),
        using="sparse",
        limit=3,
    )
    return response.points

Let's run both methods on some of the possible queries, to see if the outputs really differ.

In [10]:
queries = [
    "nutty aged cheese",
    "soft French cheese",
    "pizza ingredients",
    "a good lunch",
]

In [11]:
for query in queries:
    print("Query:", query)

    dense_results = dense_search(query)
    print("Dense Results:")
    for result in dense_results:
        print("\t-", result.payload["text"], result.score)

    sparse_results = sparse_search(query)
    print("Sparse Results:")
    for result in sparse_results:
        print("\t-", result.payload["text"], result.score)
    print()

Query: nutty aged cheese
Dense Results:
	- Mature Gouda cheese becomes grainy and develops a rich, buttery taste with extended aging. 0.58297664
	- Brie cheese features a soft, creamy interior surrounded by an edible white rind. 0.47647113
	- This French cheese has a flowing, buttery center encased in a bloomy white crust. 0.45055333
Sparse Results:
	- Aged Gouda develops a crystalline texture and nutty flavor profile after 18 months of maturation. 5.1563325
	- Mature Gouda cheese becomes grainy and develops a rich, buttery taste with extended aging. 3.0210652
	- Parmesan requires at least 12 months of cave aging to develop its signature sharp taste. 1.8819332

Query: soft French cheese
Dense Results:
	- This French cheese has a flowing, buttery center encased in a bloomy white crust. 0.6242112
	- Brie cheese features a soft, creamy interior surrounded by an edible white rind. 0.60305494
	- Croque Monsieur combines ham and Gruyère in France's answer to the toasted cheese sandwich. 0.46

## Step 5: Hybrid Search with Reciprocal Rank Fusion

Scores coming from both methods are incompatible, but RRF will not use them directly. It will only consider the order / ranking of the elements, so let's implement such a hybrid search pipeline.

In [12]:
def rrf_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=3,
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=3,
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=3,
    )
    return response.points

In [13]:
for query in queries:
    print("Query:", query)

    rrf_results = rrf_search(query)
    print("RRF Results:")
    for result in rrf_results:
        print("\t-", result.payload["text"], result.score)
    print()

Query: nutty aged cheese
RRF Results:
	- Mature Gouda cheese becomes grainy and develops a rich, buttery taste with extended aging. 0.8333334
	- Aged Gouda develops a crystalline texture and nutty flavor profile after 18 months of maturation. 0.5
	- Brie cheese features a soft, creamy interior surrounded by an edible white rind. 0.33333334

Query: soft French cheese
RRF Results:
	- This French cheese has a flowing, buttery center encased in a bloomy white crust. 1.0
	- Brie cheese features a soft, creamy interior surrounded by an edible white rind. 0.6666667
	- Grilled cheese sandwiches are the ultimate American comfort food for cold winter days. 0.25

Query: pizza ingredients
RRF Results:
	- Classic Margherita pizza topped with tomato sauce, mozzarella, and fresh basil. 1.0
	- Fresh mozzarella pairs beautifully with ripe tomatoes and basil leaves. 0.33333334
	- Croque Monsieur combines ham and Gruyère in France's answer to the toasted cheese sandwich. 0.25

Query: a good lunch
RRF Res

## Step 6: Distribution-Based Score Fusion

RRF is not the only supported fusion method. DBSF is another option that normalizes the scores of the points in each query, and sums the scores of the same point across different queries. Choosing a different algorithm can definitely impact the final outputs, so let's see how they are going to look like.

In [14]:
def dbsf_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=3,
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=3,
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.DBSF),
        limit=3,
    )
    return response.points

In [15]:
for query in queries:
    print("Query:", query)

    dbsf_results = dbsf_search(query)
    print("DBSF Results:")
    for result in dbsf_results:
        print("\t-", result.payload["text"], result.score)
    print()

Query: nutty aged cheese
DBSF Results:
	- Mature Gouda cheese becomes grainy and develops a rich, buttery taste with extended aging. 1.1558483
	- Aged Gouda develops a crystalline texture and nutty flavor profile after 18 months of maturation. 0.6808001
	- Brie cheese features a soft, creamy interior surrounded by an edible white rind. 0.43620527

Query: soft French cheese
DBSF Results:
	- This French cheese has a flowing, buttery center encased in a bloomy white crust. 1.2130752
	- Brie cheese features a soft, creamy interior surrounded by an edible white rind. 1.1703099
	- Croque Monsieur combines ham and Gruyère in France's answer to the toasted cheese sandwich. 0.30906472

Query: pizza ingredients
DBSF Results:
	- Classic Margherita pizza topped with tomato sauce, mozzarella, and fresh basil. 1.1904721
	- Fresh mozzarella pairs beautifully with ripe tomatoes and basil leaves. 0.42859802
	- Croque Monsieur combines ham and Gruyère in France's answer to the toasted cheese sandwich. 0